In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchdiffeq import odeint

from plots import plot_harmonic_oscillator, plot_kepler

torch.manual_seed(42)

## План

**Физически информированные нейронные сети**
1. Гармонический осциллятор: PINN, learning $\omega$
2. Задача Кеплера: PINN

**Нейронные ОДУ**
1. Гармонический осциллятор: Neural ODE (single $x_0, y_0$), learning $d^2x/dt^2$
2. (*) Гармонический осциллятор: Neural ODE (generalization for any initial condition), learning $d^2x/dt^2$2 field 



## PINN


Физическая информация хранится в наборе дифференциальных уравнений или других условий, накладываемых на параметры:
$$
F(x, y', y'', ... y^{(n)}) = 0,
$$
соответствующие потери можно выразить как 
$$
L_{physics} = MSE(F(x, y', y'', ... y^{(n)}), 0)
$$

Физически информированная нейронная сеть: такая, что 
$$
L(\vartheta) = L_{reg} + \lambda L_{physics}
$$
$\lambda$ вводится для балансировки вкладов. 




## Гармонический осциллятор: восстановление неизвестного параметра

$$
x''(t) + \omega^2 x(t) = 0
$$

Начальные условия: $x(0) = 1, \ x'(0) = 0$


In [2]:
def exact_solution(
        t: torch.Tensor | np.ndarray, 
        x0: float = 1., v0: float = 1, omega: float = 2.
    ) -> torch.Tensor | np.ndarray:
    """Solution for true dynamics
    d²x/dt² = -ω²x, with known x(0), v(0)
    """
    x = x0 * np.cos(omega * t) + v0 / omega * np.sin(omega * t)
    return x

OMEGA = 2.
X0, V0 = 1., 0.

### Задание 1

1. Сгенерируйте $40$ зашумленных наблюдений вдоль траектории: $(t_{\text{obs}}, x_{\text{obs}}(t) + \varepsilon), t \in [0, 10]$, $\varepsilon\sim \mathcal{N}(0, 0.04)$

In [2]:
t_obs = ... # .view(-1,1)
x_obs = ...

2. Сгенерируйте набор отложенных данных. Например, 300 наблюдений вдоль траектории: $(t_{\text{test}}, x_{\text{test}})$, на которых будет проверяться модель

In [3]:
t_test = ... # .view(-1,1)
x_test = ...

Проверим, что получилось

In [ ]:
plot_harmonic_oscillator(
    t_obs=t_obs.flatten(), x_obs=x_obs.flatten(),
    t=t_test.flatten(), x=x_test.flatten()
    )

### Задание 2 


1. Реализуйте многослойный перцептрон для решения задачи регрессии $t \rightarrow x(t)$

2. Инициализируйте модель и оптимизатор

3. Задайте функцию потерь $L_{\text{physics}}$, которая отвечает за удовлетворение $x(t)$ уравнению осциллятора. Первые и вторые производные можно рассчитать с использованием `torch.autograd.grad`. [Ссылка на документацию](docs.pytorch.org/docs/stable/generated/torch.autograd.grad.html) 

4. Сделайте параметр $\omega$ оптимизируемым с использованием `torch.nn.Parameter`, включите его в функцию потерь



In [2]:
class PINN(nn.Module):
    def __init__(self, input_dim = 1, hidden_dim = 16): 
        super(PINN, self).__init__()
        self.net = ...
    
    def forward(self, t):
        # Input: [t]
        # Output: [u]
        return self.net(t)

In [277]:
model_oscillator = ...

omega = ...
omegas = [] # to track omega

optimizer = torch.optim.Adam(list(model_oscillator.parameters())+[omega], lr=1e-3)

In [278]:
t_physics =  torch.linspace(0, 10., 50).view(-1,1).requires_grad_(True)

In [ ]:
lambda1 = 1e2

for epoch in range(0, 15_001):
    optimizer.zero_grad()

    # Physics loss based on t_physics points
    x = ... 
    dxdt = ...
    d2xdt2 = ...
    
    loss_physics = ...
   
    # Regression loss
    x = ...
    loss_reg = ...
        
    # Total loss
    loss = loss_physics + lambda1 * loss_reg

    loss.backward()
    optimizer.step()
    omegas.append(omega.item())

    if epoch % 5_000 == 0:
        with torch.no_grad():
            x = model_oscillator(t_test)

        plot_harmonic_oscillator(
            t_obs.flatten(), x_obs.flatten(),
            t_test.flatten(),  x.flatten()
        )



In [ ]:
plt.figure(figsize=(4, 2.5))
plt.plot(omegas, color='b', alpha=0.8, label="PINN estimate")
plt.hlines(OMEGA, 0, len(omegas), label="True value", color="tab:green")
plt.grid(True, alpha=0.3)
plt.xlabel("Training step")
plt.ylabel('$\omega$')
plt.legend()

plt.show()

## Задача Кеплера

В полярной системе координат траектория задается 
$$
\rho = \frac{a(1-e^2)}{1+e\cos{(\varphi)}}
$$
- $\rho$ — расстояние от тела до гравитирующего центра, 
- $ \varphi $ — истинная аномалия или угол между направлениями на перицентр орбиты и на тело, 
- $e$ — эксцентриситет,
- $a$ — большая полуось орбиты.

При замене $u = 1/\rho$ уравнение движения может быть записано как (формула Бине)
$$
\frac{d^2u}{d\varphi^2} + u = \frac{1}{a(1-e^2)}
$$
При этом
$$
\rho= \frac{1}{u} = \frac{a(1-e^2)}{1+e\cos{(\varphi)}}
$$


In [4]:
a, e = 1.5, 0.7

def exact_solution_kepler(phi, a=1.5, e=0.7):
    """
    Solution for true kepler dynamics
    """
    return  a * (1 - e**2)/(1 + e*np.cos(phi))


### Задание 1

1. Сгенерируйте 40 зашумленных наблюдений вдоль траектории $(\varphi_{\text{obs}}, \rho_{\text{obs}} + \varepsilon)$ на двух периодах вращения $\varphi \in [0, 4\pi]$. $\varepsilon \sim \mathcal{N}(0, 0.04)$. Для обучения модели будет использоваться  
$$
D = \{\varphi_{\text{obs}}, u_{\text{obs}}\}, \ \ u_{\text{obs}}=1/\rho_{\text{obs}}
$$ 

2.  Сгенерируйте ~300 примеров точной зависимости $(\varphi_{\text{test}}, u_{\text{test}})$. 



In [5]:

phi_obs = ... # .view(-1,1)
r_obs = ... 
u_obs = 1/r_obs.view(-1,1)

phi_exact = ... # .view(-1,1)
r_exact = ... 
u_exact = 1/r_exact.view(-1,1)


Проверим, что получилось

In [ ]:
plot_kepler(phi_exact, r_exact, phi_obs, r_obs)

### Задание 2

1. Дополните процедуру обучения: реализуйте регрессионные и физические потери с учетом уравнения движения. 

In [11]:
def train_pinn_kepler(
        model: nn.Module, 
        phi_obs: torch.Tensor, u_obs: torch.Tensor, 
        n_epochs: int = 5000
        ) -> nn.Module:

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    phi_physics = ... # to get gradients
    u_std = u_obs.std().item() # to balance losses 

    print("Training PINN")

    for epoch in range(n_epochs):
        optimizer.zero_grad()

        # Regression loss
        ... 
        loss_reg = ... # normilize to / u_std**2

        # Physics loss
        ... 
        loss_phys =  ...

        loss =  loss_reg + loss_phys
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if (epoch + 1) % 3000 == 0:
            print(f"Epoch {epoch+1}/{n_epochs}")
            print(f"  Loss: Total={loss.item():.6f}, Reg={loss_reg.item():.6f}, Phys={loss_phys.item():.6f}")
            print()
    
    return model


In [ ]:
model_kepler = train_pinn_kepler(...)

2. На тестовых данных 2.5 периодов проверьте сходимость модели

In [ ]:

  
plot_kepler(phi_exact, r_exact, phi_obs, r_obs, phi_predict=..., r_predict=...)

## Гармонический осциллятор: NeuralODE, restore true dynamics

$$
x''(t) + \omega^2 x(t) = 0
$$

Начальные условия: $x(0) = 1, \ x'(0) = 0$

**Цель:** восстановить настоящую динамику, т.е. $x''(t) = - \omega^2 x(t)$


## Задание 1

1. Задайте истинную динамику системы, которая по входному состоянию $(x, dx/dt)$ рассчитывает $dx/dt, d^2x/dt^2$.

2. Задайте начальное состояние системы $(x_0, v_0)$. С помощью `torchdiffeq.odeint` получите несколько (~30) наблюдений вдоль траектории $\{x_{\text{obs}}(t), v_{\text{obs}}(t)\}$. Добавьте шум к измерениям $x, v$


In [23]:
def true_dynamics(t: torch.Tensor, state) -> torch.Tensor:
    """
    params:
        t: time
        state: [x, dx/dt], Tensor[N, 2]
    returns:
        Tensor[N, 2]
    """
    global OMEGA
    pass # dx/dt, d2x/dt2

In [30]:
t_span = torch.linspace(0, 10., 30)
initial_state = torch.tensor([X0, V0])

true_traj = ...
obs_traj = ...

t_test = ...
traj_test = ...

In [ ]:
plot_harmonic_oscillator(
    t_span, obs_traj[:, 0], t_test, traj_test[:, 0]
)

### Задание 2

1. Реализуйте многослойный перцептрон - Нейронное ОДУ $f()$: 
$$
(x(t_i), v(t_i)) \overset{f()}{\rightarrow} ((dx/dt)(t_i), (d^2x/dt^2)(t_i))
$$

In [34]:
class NeuralODE(nn.Module):
    def __init__(self): 
        super(NeuralODE, self).__init__()
        self.net = ...
    
    def forward(self, t, state): # t parameter is only for odeint
        # Input: [t, (x, x')]
        # Output: [x', x'']
        return ... 

2. Реализуйте процедуру обучения модели. Потери задаются таким образом, чтобы интегрированные траектории совпадали. Траектории можно интегрировать с использованием `odeint(f, initial_condition, t)`

In [ ]:
model_node = NeuralODE(...)
... 

print("Training Neural ODE")
for epoch in range(1000):
    ...
    
    if epoch % 300 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")
        ...

        print('--- x(t) ---')
        plot_harmonic_oscillator(
            t_span.flatten(), obs_traj[:, 0].flatten(),
            t=t_test.flatten(), x=... 
        )
        print('--- dx/dt ---')
        plot_harmonic_oscillator(
            t_span.flatten(), obs_traj[:, 1].flatten(), 
            t=t_test.flatten(), x=...
        )

3. Для проверки постройте траектории с другими НУ.

In [ ]:
new_x0, new_v0 = 0.5, 1.0
new_initial = torch.tensor([new_x0, new_v0])

new_true = ...
new_pred = ...

plot_harmonic_oscillator(
    t_span, new_true[:, 0], t=t_span, x=new_pred[:, 0] # plot x(t)
) # why?

4. Сравните динамику, выученную нейронной сетью, с реальной динамикой для задачи гармонического осциллятора $d^2x/dt^2 = -\omega^2 x$. Для этого нужно учесть, что нас интересует динамика вдоль траектории с заданным ранее НУ. 

    - сгенерируйте траекторию  $(x(t), v(t))$
    - в каждой точке вдоль траектории получите $d^2x/dt^2 (t)$ с помощью НС
    - сравните с теоретическим $d^2x/dt^2 (t)$


In [55]:
def exact_solution_xv(
        t: torch.Tensor | np.ndarray, 
        x0: float = 1., v0: float = 1, omega: float = 2.
    ) -> torch.Tensor | np.ndarray:
    """Solution for true dynamics
    d²x/dt² = -ω²x, with known x(0), v(0)
    """
    x = x0 * torch.cos(omega * t) + v0 / omega * torch.sin(omega * t)
    v = x0 * torch.sin(omega * t) - v0 * torch.cos(omega * t)
    return torch.stack([x, v]).T

In [ ]:
t = torch.linspace(0, 10., 300)

with torch.no_grad():
    given_trajectory = ...
    dvdt_pred = ...
    

dvdt_true = ...
dvdt_pred = ...


plt.figure(figsize=(5, 2))
plt.plot(t.numpy(), dvdt_true.numpy(), 'r-',  linewidth=1, alpha=0.6, label='True: -ω²x')
plt.plot(t.numpy(), dvdt_pred.numpy(), 'k-', linewidth=1, label='Model: dv/dt')
plt.xlabel('Time')
plt.ylabel('dv/dt')
plt.title('Acceleration Component')
plt.legend()
plt.grid(True)